In [3]:
import torch
torch.set_default_dtype(torch.double)

In [7]:
def f(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
	return torch.sin(x).prod() + torch.cos(y).prod()

In [27]:
parameters = (torch.rand(6), torch.rand(6))
param_indices = [0]
for param in parameters:
	param.requires_grad = True
	param_indices.append(param_indices[-1] + param.numel())
hessian = torch.zeros(param_indices[-1], param_indices[-1], dtype=torch.double)
print(param_indices, hessian.shape, sep="\n")
val = f(*parameters)
for iParam, param in enumerate(parameters):
	param.grad = torch.autograd.grad(val, param, None, True, True, True, True, False, True)[0]
	assert param.grad is not None
	for iGrad, grad_elm in enumerate(param.grad.reshape(-1)):
		for jParam, param_for_grad in enumerate(parameters):
			hessian[param_indices[iParam] + iGrad, param_indices[jParam]:param_indices[jParam + 1]] = torch.autograd.grad(grad_elm, param_for_grad, None, True, False, True, True, False, True)[0] # param.numel() - by - param.numel()
hessian = (hessian + hessian.T) / 2.0
print(parameters, tuple(param.grad for param in parameters), hessian, sep='\n')

[0, 6, 12]
torch.Size([12, 12])
(tensor([0.5530, 0.8362, 0.7517, 0.3880, 0.4656, 0.8847], requires_grad=True), tensor([0.8436, 0.5904, 0.2038, 0.8974, 0.5246, 0.0686], requires_grad=True))
(tensor([0.0567, 0.0316, 0.0374, 0.0856, 0.0696, 0.0286],
       grad_fn=<MulBackward0>), tensor([-0.3273, -0.1952, -0.0602, -0.3650, -0.1685, -0.0200],
       grad_fn=<MulBackward0>))
tensor([[-0.0350,  0.0512,  0.0606,  0.1387,  0.1128,  0.0464,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0512, -0.0350,  0.0338,  0.0773,  0.0629,  0.0259,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0606,  0.0338, -0.0350,  0.0916,  0.0745,  0.0306,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.1387,  0.0773,  0.0916, -0.0350,  0.1703,  0.0701,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.1128,  0.0629,  0.0745,  0.1703, -0.0350,  0.0570,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  

In [29]:
print(torch.linalg.cond(hessian))
precond: torch.Tensor = (1.0 / torch.norm(hessian, torch.inf, 0).sqrt()).diag()
hessian_precond = precond @ hessian @ precond
hessian_precond = (hessian_precond + hessian_precond.T) / 2.0
print(torch.linalg.cond(hessian_precond))

tensor(11.7011, grad_fn=<SqueezeBackward1>)
tensor(3.3960, grad_fn=<SqueezeBackward1>)
